# NancyBot - Phase 3: Simple Backtesting

**Objective**: Implement and test a naive replication strategy - buy on disclosure, hold for fixed period.

**Date**: 2026-03-05

---

## Overview

This notebook implements a baseline strategy:
- **Entry**: Buy stock at next market open after disclosure date
- **Position Sizing**: Equal weight allocation
- **Hold Period**: Fixed (30, 60, or 90 days)
- **Exit**: Sell after hold period
- **Benchmark**: Compare to S&P 500 buy-and-hold

This establishes our baseline performance before adding smart execution criteria in Phase 4.

## Setup

In [13]:
# Standard imports
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Add src directory to path
sys.path.append('../src')

# Import custom modules
from data_loader import CongressionalTradeLoader
from portfolio import Portfolio
from metrics import calculate_comprehensive_metrics, print_metrics

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)

# Plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("Setup complete!")

Setup complete!


## 1. Load Data

In [14]:
# Load trade data
purchases = pd.read_csv('../data/pelosi_purchases_with_returns.csv',
                       parse_dates=['trade_date', 'disclosure_date', 'filing_date'])

print(f"Loaded {len(purchases)} purchase transactions")
print(f"Date range: {purchases['disclosure_date'].min().date()} to {purchases['disclosure_date'].max().date()}")
print(f"Unique tickers: {purchases['ticker'].nunique()}")

purchases.head()

Loaded 18 purchase transactions
Date range: 2020-02-22 to 2026-02-27
Unique tickers: 5


,trade_id,trade_date,disclosure_date,filing_date,delay_days,ticker,transaction_type,amount_range,amount_min,amount_max,amount_midpoint,asset_type,owner,representative,party,state,trade_price,disclosure_price,price_change_during_delay,price_change_pct_delay,forward_return_1d,forward_return_5d,forward_return_20d,forward_return_60d
0,TRADE_000000,2020-01-15,2020-02-22,2020-02-22,38,AAPL,Purchase,"$500,000 - $1,000,000",500000,1000000,750000.0,Stock,Spouse,Nancy Pelosi,Democrat,California,75.049683,72.048004,-3.001678,-3.999588,-3.387206,0.211289,-24.753502,5.301417
1,TRADE_000001,2020-03-20,2020-04-25,2020-04-25,36,MSFT,Purchase,"$1,000,000 - $5,000,000",1000000,5000000,3000000.0,Stock,Spouse,Nancy Pelosi,Democrat,California,130.469696,165.331284,34.861588,26.720065,-2.436090,2.752043,4.611131,21.999241
2,TRADE_000002,2020-06-15,2020-07-30,2020-07-30,45,NVDA,Purchase,"$250,000 - $500,000",250000,500000,375000.0,Stock,Spouse,Nancy Pelosi,Democrat,California,9.142588,10.577946,1.435358,15.699691,0.007041,6.797619,18.977279,28.079091
3,TRADE_000003,2021-01-21,2021-03-01,2021-03-01,39,TSLA,Purchase,"$500,000 - $1,000,000",500000,1000000,750000.0,Stock,Spouse,Nancy Pelosi,Democrat,California,281.663330,239.476669,-42.186661,-14.977690,-4.452764,-21.634674,-14.913074,-15.831743
4,TRADE_000004,2021-03-19,2021-05-02,2021-05-02,44,GOOGL,Purchase,"$1,000,000 - $5,000,000",1000000,5000000,3000000.0,Stock,Spouse,Nancy Pelosi,Democrat,California,100.587990,116.275452,15.687462,15.595760,-1.547106,-2.190702,1.626073,16.166760


In [15]:
# Load price data for all tickers
tickers = purchases['ticker'].unique().tolist()
start_date = purchases['disclosure_date'].min() - timedelta(days=30)
end_date = datetime.now()

print(f"Downloading price data for {len(tickers)} tickers...")
print(f"Date range: {start_date.date()} to {end_date.date()}")

loader = CongressionalTradeLoader()
price_data = loader.load_price_data(
    # tickers=tickers,
    # start_date=start_date.strftime('%Y-%m-%d'),
    # end_date=end_date.strftime('%Y-%m-%d')
    tickers=tickers,
    start_date=start_date.strftime('%Y-%m-%d'),
    end_date=end_date.strftime('%Y-%m-%d'),
    batch_mode=True,  # Default: downloads all at once (recommended)
    delay=3.0,         # Delay for individual fallback mode if needed
    max_retries=3      # Retry with exponential backoff if rate limited
)

print(f"\nSuccessfully loaded price data for {len(price_data)} tickers")

Date range: 2020-01-23 to 2026-03-05
Using batch mode (recommended to avoid rate limits)...



5 Failed downloads:
['AAPL', 'GOOGL', 'NVDA', 'TSLA', 'MSFT']: YFRateLimitError('Too Many Requests. Rate limited. Try after a while.')


  ✗ AAPL (no valid data)
  ✗ MSFT (no valid data)
  ✗ NVDA (no valid data)
  ✗ TSLA (no valid data)
  ✗ GOOGL (no valid data)

Batch download complete: 0/5 tickers

Successfully loaded 0/5 tickers

Successfully loaded price data for 0 tickers


In [16]:
# Download S&P 500 benchmark data
print("Downloading S&P 500 (SPY) benchmark data...")
spy_data = yf.download('SPY', start=start_date, end=end_date, progress=False)
if isinstance(spy_data.columns, pd.MultiIndex):
    spy_data.columns = spy_data.columns.get_level_values(0)

print(f"Loaded {len(spy_data)} days of SPY data")


1 Failed download:
['SPY']: YFRateLimitError('Too Many Requests. Rate limited. Try after a while.')


Loaded 0 days of SPY data


## 2. Define Naive Replication Strategy

In [17]:
def naive_replication_strategy(purchases_df, price_data, hold_days=60, max_positions=10, initial_capital=100000):
    """
    Naive replication strategy:
    - Buy at next open after disclosure date
    - Hold for fixed number of days
    - Equal weight allocation
    
    Args:
        purchases_df: DataFrame with purchase transactions
        price_data: Dictionary of price DataFrames by ticker
        hold_days: Number of days to hold position
        max_positions: Maximum concurrent positions
        initial_capital: Starting capital
        
    Returns:
        portfolio: Portfolio object with results
    """
    portfolio = Portfolio(initial_capital=initial_capital)
    
    # Track open positions and their exit dates
    open_positions = {}  # ticker -> exit_date
    
    # Sort trades by disclosure date
    trades_sorted = purchases_df.sort_values('disclosure_date').copy()
    
    # Track all dates for equity curve
    all_dates = set()
    for df in price_data.values():
        all_dates.update(df.index)
    all_dates = sorted(all_dates)
    
    # Process each trade
    for idx, trade in trades_sorted.iterrows():
        ticker = trade['ticker']
        disclosure_date = trade['disclosure_date']
        
        # Skip if no price data
        if ticker not in price_data:
            continue
        
        ticker_prices = price_data[ticker]
        
        # Find next available date after disclosure
        future_dates = ticker_prices.index[ticker_prices.index > disclosure_date]
        if len(future_dates) == 0:
            continue
        
        entry_date = future_dates[0]
        entry_price = ticker_prices.loc[entry_date, 'Open']
        
        # Check if we already have this position
        if portfolio.has_position(ticker):
            continue
        
        # Check if we're at max positions
        if len(portfolio.positions) >= max_positions:
            continue
        
        # Calculate position size (equal weight)
        target_position_value = portfolio.get_total_value() / max_positions
        shares = target_position_value / entry_price
        
        # Buy
        success = portfolio.buy(
            ticker=ticker,
            price=entry_price,
            shares=shares,
            date=entry_date,
            notes=f"Disclosure: {disclosure_date.date()}"
        )
        
        if success:
            # Calculate exit date
            exit_idx = ticker_prices.index.get_loc(entry_date) + hold_days
            if exit_idx < len(ticker_prices):
                exit_date = ticker_prices.index[exit_idx]
                open_positions[ticker] = exit_date
        
        # Check for positions to exit
        positions_to_close = []
        for pos_ticker, exit_date in open_positions.items():
            if entry_date >= exit_date:
                positions_to_close.append(pos_ticker)
        
        # Close positions
        for pos_ticker in positions_to_close:
            if pos_ticker in price_data:
                pos_prices = price_data[pos_ticker]
                exit_date = open_positions[pos_ticker]
                
                # Find actual exit price
                if exit_date in pos_prices.index:
                    exit_price = pos_prices.loc[exit_date, 'Close']
                else:
                    # Find next available date
                    future = pos_prices.index[pos_prices.index >= exit_date]
                    if len(future) > 0:
                        exit_price = pos_prices.loc[future[0], 'Close']
                    else:
                        exit_price = pos_prices['Close'].iloc[-1]
                
                portfolio.sell(
                    ticker=pos_ticker,
                    price=exit_price,
                    shares=None,  # Sell all
                    date=exit_date,
                    notes=f"Hold period: {hold_days} days"
                )
            
            del open_positions[pos_ticker]
    
    # Close any remaining positions
    for ticker in list(portfolio.positions.keys()):
        if ticker in price_data:
            exit_price = price_data[ticker]['Close'].iloc[-1]
            exit_date = price_data[ticker].index[-1]
            
            portfolio.sell(
                ticker=ticker,
                price=exit_price,
                shares=None,
                date=exit_date,
                notes="Final close"
            )
    
    # Build equity curve
    equity_dates = [d for d in all_dates if d >= trades_sorted['disclosure_date'].min()]
    
    for date in equity_dates:
        # Update prices
        current_prices = {}
        for ticker in portfolio.positions.keys():
            if ticker in price_data:
                ticker_df = price_data[ticker]
                if date in ticker_df.index:
                    current_prices[ticker] = ticker_df.loc[date, 'Close']
                else:
                    # Use last available price
                    past_dates = ticker_df.index[ticker_df.index <= date]
                    if len(past_dates) > 0:
                        current_prices[ticker] = ticker_df.loc[past_dates[-1], 'Close']
        
        portfolio.update_prices(current_prices, date)
    
    return portfolio

## 3. Run Backtest with Different Hold Periods

In [18]:
# Test different hold periods
hold_periods = [30, 60, 90]
results = {}

for hold_days in hold_periods:
    print(f"\n{'='*80}")
    print(f"Running backtest with {hold_days}-day hold period...")
    print(f"{'='*80}")
    
    portfolio = naive_replication_strategy(
        purchases_df=purchases,
        price_data=price_data,
        hold_days=hold_days,
        max_positions=10,
        initial_capital=100000
    )
    
    results[hold_days] = portfolio
    
    # Print summary
    portfolio.print_summary()

print("\n" + "="*80)
print("All backtests complete!")
print("="*80)


Running backtest with 30-day hold period...
PORTFOLIO SUMMARY
Initial Capital:     $     100,000.00
Current Value:       $     100,000.00
  Cash:              $     100,000.00
  Positions:         $           0.00
Total Return:        $           0.00 ( +0.00%)

Current Positions:                 0
Total Trades:                      0
Closed Trades:                     0
  Winners:                         0
  Losers:                          0
Win Rate:                       0.0%

Running backtest with 60-day hold period...
PORTFOLIO SUMMARY
Initial Capital:     $     100,000.00
Current Value:       $     100,000.00
  Cash:              $     100,000.00
  Positions:         $           0.00
Total Return:        $           0.00 ( +0.00%)

Current Positions:                 0
Total Trades:                      0
Closed Trades:                     0
  Winners:                         0
  Losers:                          0
Win Rate:                       0.0%

Running backtest with 90-da

## 4. Calculate Comprehensive Metrics

In [19]:
# Calculate metrics for each strategy
all_metrics = {}

for hold_days, portfolio in results.items():
    equity_curve = portfolio.get_equity_curve_df()
    trade_history = portfolio.get_trade_history_df()
    summary = portfolio.get_summary()
    
    if len(equity_curve) > 0:
        days = (equity_curve['date'].max() - equity_curve['date'].min()).days
    else:
        days = 365
    
    # Calculate benchmark returns for same period
    if len(equity_curve) > 0:
        start_date = equity_curve['date'].min()
        end_date = equity_curve['date'].max()
        
        spy_period = spy_data[(spy_data.index >= start_date) & (spy_data.index <= end_date)]
        benchmark_returns = spy_period['Close'].pct_change().dropna()
    else:
        benchmark_returns = None
    
    metrics = calculate_comprehensive_metrics(
        portfolio_summary=summary,
        equity_curve=equity_curve,
        trade_history=trade_history,
        days=days,
        benchmark_returns=benchmark_returns
    )
    
    all_metrics[hold_days] = metrics
    
    print(f"\n{'='*80}")
    print(f"{hold_days}-DAY HOLD PERIOD METRICS")
    print(f"{'='*80}")
    print_metrics(metrics)

KeyError: 'total_value'

## 5. Compare Hold Periods

In [ ]:
# Create comparison table
comparison_df = pd.DataFrame(all_metrics).T
comparison_df.index.name = 'Hold Period (days)'

print("\n" + "="*80)
print("HOLD PERIOD COMPARISON")
print("="*80)

# Select key metrics for comparison
key_metrics = [
    'total_return', 'annualized_return', 'volatility', 'sharpe_ratio',
    'max_drawdown', 'win_rate', 'profit_factor', 'closed_trades'
]

if 'alpha' in comparison_df.columns:
    key_metrics.insert(2, 'alpha')

print(comparison_df[key_metrics].round(2))

In [ ]:
# Visualize comparison
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

# Plot 1: Total Return
comparison_df['total_return'].plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Total Return', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Return (%)')
axes[0].set_xlabel('Hold Period (days)')
axes[0].grid(alpha=0.3)
axes[0].axhline(0, color='red', linestyle='--', alpha=0.5)

# Plot 2: Sharpe Ratio
comparison_df['sharpe_ratio'].plot(kind='bar', ax=axes[1], color='green')
axes[1].set_title('Sharpe Ratio', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Sharpe Ratio')
axes[1].set_xlabel('Hold Period (days)')
axes[1].grid(alpha=0.3)
axes[1].axhline(0, color='red', linestyle='--', alpha=0.5)

# Plot 3: Max Drawdown
comparison_df['max_drawdown'].plot(kind='bar', ax=axes[2], color='red')
axes[2].set_title('Maximum Drawdown', fontsize=12, fontweight='bold')
axes[2].set_ylabel('Drawdown (%)')
axes[2].set_xlabel('Hold Period (days)')
axes[2].grid(alpha=0.3)

# Plot 4: Win Rate
comparison_df['win_rate'].plot(kind='bar', ax=axes[3], color='orange')
axes[3].set_title('Win Rate', fontsize=12, fontweight='bold')
axes[3].set_ylabel('Win Rate (%)')
axes[3].set_xlabel('Hold Period (days)')
axes[3].grid(alpha=0.3)
axes[3].axhline(50, color='gray', linestyle='--', alpha=0.5)

# Plot 5: Profit Factor
comparison_df['profit_factor'].plot(kind='bar', ax=axes[4], color='purple')
axes[4].set_title('Profit Factor', fontsize=12, fontweight='bold')
axes[4].set_ylabel('Profit Factor')
axes[4].set_xlabel('Hold Period (days)')
axes[4].grid(alpha=0.3)
axes[4].axhline(1, color='red', linestyle='--', alpha=0.5)

# Plot 6: Closed Trades
comparison_df['closed_trades'].plot(kind='bar', ax=axes[5], color='teal')
axes[5].set_title('Number of Closed Trades', fontsize=12, fontweight='bold')
axes[5].set_ylabel('Trades')
axes[5].set_xlabel('Hold Period (days)')
axes[5].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Plot Equity Curves

In [ ]:
# Plot equity curves for all hold periods
fig, ax = plt.subplots(figsize=(15, 8))

colors = ['steelblue', 'green', 'orange']

for i, (hold_days, portfolio) in enumerate(results.items()):
    equity_curve = portfolio.get_equity_curve_df()
    
    if len(equity_curve) > 0:
        # Normalize to starting value
        equity_curve['normalized_value'] = (equity_curve['total_value'] / portfolio.initial_capital) * 100
        
        ax.plot(equity_curve['date'], equity_curve['normalized_value'],
               label=f'{hold_days}-day hold', linewidth=2, color=colors[i], alpha=0.8)

# Add SPY benchmark
if len(spy_data) > 0:
    spy_normalized = (spy_data['Close'] / spy_data['Close'].iloc[0]) * 100
    ax.plot(spy_data.index, spy_normalized, 
           label='S&P 500 (SPY)', linewidth=2, color='black', linestyle='--', alpha=0.6)

ax.set_title('Equity Curve Comparison (Normalized to 100)', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Portfolio Value (Normalized)')
ax.legend(loc='best')
ax.grid(alpha=0.3)
ax.axhline(100, color='gray', linestyle=':', alpha=0.5)

plt.tight_layout()
plt.show()

## 7. Trade Analysis

In [ ]:
# Analyze trades for best performing strategy
best_hold_period = max(all_metrics.items(), key=lambda x: x[1]['total_return'])[0]
best_portfolio = results[best_hold_period]

print(f"\n{'='*80}")
print(f"DETAILED TRADE ANALYSIS - Best Strategy ({best_hold_period}-day hold)")
print(f"{'='*80}")

trade_history = best_portfolio.get_trade_history_df()

# Show sample of trades
print("\nSample of Executed Trades:")
print(trade_history[['date', 'action', 'ticker', 'shares', 'price', 'value']].head(20))

# Analyze closed positions
closed_trades = trade_history[trade_history['action'] == 'SELL'].copy()

if len(closed_trades) > 0:
    print(f"\n\nClosed Trades Statistics:")
    print(f"Total closed: {len(closed_trades)}")
    print(f"\nP&L Distribution:")
    print(closed_trades['pnl_pct'].describe())
    
    # Top winners
    print("\nTop 5 Winners:")
    top_winners = closed_trades.nlargest(5, 'pnl_pct')[['date', 'ticker', 'pnl', 'pnl_pct']]
    print(top_winners.to_string(index=False))
    
    # Top losers
    print("\nTop 5 Losers:")
    top_losers = closed_trades.nsmallest(5, 'pnl_pct')[['date', 'ticker', 'pnl', 'pnl_pct']]
    print(top_losers.to_string(index=False))

## 8. Key Findings & Next Steps

In [ ]:
print("=" * 80)
print("KEY FINDINGS FROM NAIVE REPLICATION STRATEGY")
print("=" * 80)

best_metrics = all_metrics[best_hold_period]

print(f"\n1. BASELINE PERFORMANCE")
print(f"   - Best hold period: {best_hold_period} days")
print(f"   - Total return: {best_metrics['total_return']:.2f}%")
print(f"   - Annualized return: {best_metrics['annualized_return']:.2f}%")
print(f"   - Sharpe ratio: {best_metrics['sharpe_ratio']:.2f}")

if 'alpha' in best_metrics:
    print(f"\n2. BENCHMARK COMPARISON")
    print(f"   - S&P 500 return: {best_metrics.get('benchmark_return', 0):.2f}%")
    print(f"   - Alpha: {best_metrics.get('alpha', 0):+.2f}%")
    
    if best_metrics.get('alpha', 0) > 0:
        print("   → Strategy OUTPERFORMS benchmark")
    else:
        print("   → Strategy UNDERPERFORMS benchmark")

print(f"\n3. RISK METRICS")
print(f"   - Max drawdown: {best_metrics['max_drawdown']:.2f}%")
print(f"   - Volatility: {best_metrics['volatility']:.2f}%")

print(f"\n4. TRADING STATS")
print(f"   - Win rate: {best_metrics['win_rate']:.1f}%")
print(f"   - Profit factor: {best_metrics['profit_factor']:.2f}")
print(f"   - Total trades: {best_metrics['closed_trades']:.0f}")

print(f"\n5. NEXT STEPS FOR PHASE 4")
print(f"   a) Implement smart execution criteria to improve performance")
print(f"   b) Filter out trades that already ran up during delay")
print(f"   c) Add momentum and volatility filters")
print(f"   d) Consider trade size as conviction signal")
print(f"   e) Optimize position sizing")

print("\n" + "="*80)
print("PHASE 3 COMPLETE - Ready for Phase 4: Execution Criteria")
print("="*80)